In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
cd drive/MyDrive/Colab_project/model_trained_MNIST/ResNet_for_MNIST


In [ ]:
!pip install pytorch-lightning wandb matplotlib scipy
!pip install torchvision tqdm

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.io import loadmat
import random
import argparse

import pytorch_lightning as pl
import torch
from torch import nn, optim
import torch.nn.functional as F
from torchmetrics import Accuracy
import torchvision
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, WeightedRandomSampler, Subset, Dataset
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint, LearningRateMonitor
from pytorch_lightning.loggers import WandbLogger
import pathlib
import wandb

from torch import multiprocessing as mp
mp.set_start_method('spawn', force=True)

import warnings
warnings.filterwarnings("ignore", category=UserWarning)
import os
os.environ['PYTHONWARNINGS'] = "ignore"




class LitResNet18(pl.LightningModule):
    def __init__(self, num_classes=10, lr=1e-4):
        super(LitResNet18, self).__init__()
        self.save_hyperparameters()

        self.model = models.resnet18(weights=None)

        self.model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)

        self.model.fc = nn.Linear(self.model.fc.in_features, num_classes)

        self.accuracy = Accuracy(num_classes=num_classes, task='multiclass')

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        inputs, labels = batch
        outputs = self(inputs)
        loss = nn.CrossEntropyLoss()(outputs, labels)
        acc = self.accuracy(outputs.softmax(dim=-1), labels)

        self.log('train_loss', loss, on_step=True, on_epoch=True, prog_bar=True, logger=True)
        self.log('train_acc', acc, on_step=True, on_epoch=True, prog_bar=True, logger=True)
        return loss

    def validation_step(self, batch, batch_idx):
        inputs, labels = batch
        outputs = self(inputs)
        loss = nn.CrossEntropyLoss()(outputs, labels)
        acc = self.accuracy(outputs.softmax(dim=-1), labels)

        self.log('val_loss', loss, on_step=True, on_epoch=True, prog_bar=True, logger=True)
        self.log('val_acc', acc, on_step=True, on_epoch=True, prog_bar=True, logger=True)
        return loss

    def configure_optimizers(self):

        optimizer = optim.AdamW(self.parameters(), lr=self.hparams.lr, weight_decay=1e-4)

        scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=0.5, patience=2
        )

        lr_scheduler_config = {
            'scheduler': scheduler,
            'monitor': 'val_loss',
            'interval': 'epoch',
            'frequency': 1
        }
        return [optimizer], [lr_scheduler_config]



In [ ]:
import torch
import torch.nn.functional as F
import pandas as pd
import numpy as np
import os
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

def get_test_loader(batch_size=256, img_size=224):

    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,))
    ])
    dataset = datasets.MNIST(root='./data', train=False, transform=transform, download=True)
    return DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=2)




def load_model_weights(model, checkpoint_path, device):

    try:

        checkpoint = torch.load(checkpoint_path, map_location='cpu')

        # Extract state_dict
        if isinstance(checkpoint, dict) and 'state_dict' in checkpoint:
            state_dict = checkpoint['state_dict']
        elif isinstance(checkpoint, dict) and 'model' in checkpoint:
             state_dict = checkpoint['model']
        else:
            state_dict = checkpoint # Assume direct state_dict

        # Load
        model.load_state_dict(state_dict, strict=False)
        model.to(device)
        model.eval()
        return True
    except Exception as e:
        print(f"Error loading {checkpoint_path}: {e}")
        return False



def find_optimal_noise(model_folder, noise_range, target_acc, ModelClass, device, num_calib_instances=5):
    print(f"\n=== PHASE 1: Calibration (Target Accuracy: {target_acc*100:.1f}%) ===")

    test_loader = get_test_loader(batch_size=512)
    avg_accs = []

    available_instances = []
    for i in range(60):
        # NOTE: Change filename pattern here if testing ResNet or VGG
        path = os.path.join(model_folder, f"resnet-224-{i}-final.pt")
        if os.path.exists(path):
            available_instances.append(i)

    calib_instances = available_instances[:num_calib_instances]
    print(f"Calibrating using instances: {calib_instances}")

    for noise in noise_range:
        current_noise_accs = []
        for inst_idx in calib_instances:
            path = os.path.join(model_folder, f"resnet-224-{inst_idx}-final.pt")
            model = ModelClass(num_classes=10)
            if not load_model_weights(model, path, device): continue

            correct = 0; total = 0
            with torch.no_grad():
                for imgs, lbls in test_loader:
                    imgs = imgs.to(device)
                    lbls = lbls.to(device)
                    imgs = F.interpolate(imgs, size=(224, 224), mode='bilinear', align_corners=False)

                    if noise > 0:
                        imgs += torch.randn_like(imgs) * noise


                    logits = model(imgs)
                    preds = logits.argmax(dim=1)
                    correct += (preds == lbls).sum().item()
                    total += lbls.size(0)

            current_noise_accs.append(correct/total)

        mean_acc = np.mean(current_noise_accs)
        avg_accs.append(mean_acc)
        print(f"Noise {noise:.2f} -> Mean Acc: {mean_acc*100:.2f}%")

    best_idx = np.argmin(np.abs(np.array(avg_accs) - target_acc))
    best_noise = noise_range[best_idx]
    best_real_acc = avg_accs[best_idx]

    print(f">>> Optimal Noise Level Found: {best_noise:.2f} (Acc: {best_real_acc*100:.2f}%)")
    return best_noise


def generate_behavior_data(model_folder, output_folder, noise_level, ModelClass, device):
    print(f"\n=== PHASE 2: Generating Data at Noise {noise_level:.2f} ===")

    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    test_loader = get_test_loader(batch_size=256)

    for inst_idx in range(60):
        filename = f"resnet-224-{inst_idx}-final.pt"
        ckpt_path = os.path.join(model_folder, filename)
        if not os.path.exists(ckpt_path): continue

        print(f"Processing Instance {inst_idx}...", end="\r")
        model = ModelClass(num_classes=10)
        load_model_weights(model, ckpt_path, device)

        data_rows = []
        with torch.no_grad():
            for batch_i, (imgs, lbls) in enumerate(test_loader):
                imgs = imgs.to(device)

                imgs = F.interpolate(imgs, size=(224, 224), mode='bilinear', align_corners=False)


                if noise_level > 0:
                    imgs += torch.randn_like(imgs) * noise_level


                logits = model(imgs)

                mean_logits = logits.mean(dim=1, keepdim=True)
                std_logits = logits.std(dim=1, keepdim=True) + 1e-8
                norm_logits = (logits - mean_logits) / std_logits

               
                top2_vals = torch.topk(norm_logits, k=2, dim=1).values
                conf_top2_batch = (top2_vals[:, 0] - top2_vals[:, 1]).cpu().numpy()

                preds = torch.argmax(norm_logits, dim=1).cpu().numpy()
                lbls_np = lbls.numpy()

                logits_np = logits.cpu().numpy()
                norm_logits_np = norm_logits.cpu().numpy()

                for i in range(len(lbls_np)):
                    curr_norm_logits = norm_logits_np[i]

                    row = {
                        'instance_id': inst_idx,
                        'image_index': batch_i * 256 + i,
                        'true_label': lbls_np[i],
                        'noise_level': noise_level,
                        'prediction': preds[i],
                        'correct': 1 if preds[i] == lbls_np[i] else 0,
                        'conf_top2': conf_top2_batch[i] # Now properly mapped to Z-score scale
                    }

                    for d in range(10):
                        row[f'logit_{d}'] = logits_np[i, d]
                        row[f'norm_logit_{d}'] = curr_norm_logits[d]

                    data_rows.append(row)

        df = pd.DataFrame(data_rows)
        save_name = f"behavior_resnet_inst{inst_idx}_noise{noise_level:.2f}.csv"
        df.to_csv(os.path.join(output_folder, save_name), index=False)

    print(f"\nDone! All files saved to: {output_folder}")


def run_full_analysis(model_folder, output_folder, target_acc, ModelClass):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    noise_range = np.arange(0.2, 0.25, 0.01)

    best_noise = find_optimal_noise(model_folder, noise_range, target_acc, ModelClass, device)
    generate_behavior_data(model_folder, output_folder, best_noise, ModelClass, device)

In [ ]:
run_full_analysis(
    model_folder = "/content/drive/MyDrive/Colab_project/model_trained_MNIST/ANN_training_script_final/Model_Weight/model_resnet_224/",
    output_folder = "/content/drive/MyDrive/Colab_project/model_trained_MNIST/ANN_training_script_final/Model_test_result_resize_addnoise/ResNet",
    target_acc = 0.64,
    ModelClass = LitResNet18
)